<a href="https://colab.research.google.com/github/SanskarSontakke/gw150914-experience/blob/main/gw150914.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install numpy scipy librosa resampy gwpy==4.0.1 pycbc==2.11.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.7/168.7 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.0/115.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.1/203.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 5

In [ ]:
import numpy as np
import scipy.signal
import scipy.signal.windows
import scipy.io.wavfile
import scipy.fft
import json
import logging
import math
import sys

# --- Dependency Checks ---
try:
    import librosa
except ImportError:
    print("ERROR: librosa not found. Please install via: !pip install librosa")
    sys.exit(1)

try:
    import resampy
except ImportError:
    print("ERROR: resampy not found. Please install via: !pip install resampy")
    sys.exit(1)

try:
    from gwpy.timeseries import TimeSeries
except ImportError:
    print("ERROR: gwpy not found. Please install via: !pip install gwpy")
    sys.exit(1)

# Configure detailed but uncluttered logging with timestamps
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s - %(message)s',
    datefmt='%H:%M:%S'
)

def process_gw_data():
    try:
        # =====================================================================
        # PHASE 1: DATA EXTRACTION & CONDITIONING
        # =====================================================================
        logging.info("--- PHASE 1: DATA EXTRACTION & CONDITIONING ---")

        gps_merger_time = 1126259462.4
        duration = 32  # 32 seconds total
        t_start = gps_merger_time - (duration / 2)
        t_end = gps_merger_time + (duration / 2)

        logging.info(f"Fetching H1 strain data | GPS: {t_start} to {t_end}")
        strain = TimeSeries.fetch_open_data('H1', t_start, t_end)
        fs = int(strain.sample_rate.value)

        # Safeguard: Convert any potential missing data (NaN) from the detector to 0.0
        strain_data = np.nan_to_num(strain.value)
        N = len(strain_data)
        logging.info(f"Data fetched | Sample Rate: {fs} Hz | Total Samples: {N}")

        logging.info("Applying Tukey window (alpha=0.1) to mitigate spectral leakage...")
        window = scipy.signal.windows.tukey(N, alpha=0.1)
        strain_windowed = strain_data * window

        logging.info(f"Computing PSD (Welch's method) | Segment Length: 4.0s ({4 * fs} samples)")
        nperseg = int(4 * fs)
        f_psd, psd = scipy.signal.welch(strain_windowed, fs=fs, nperseg=nperseg)

        logging.info("Whitening strain data in frequency domain...")
        strain_fft = scipy.fft.rfft(strain_windowed)
        freqs = scipy.fft.rfftfreq(N, 1.0 / fs)

        psd_interp = np.interp(freqs, f_psd, psd)
        # SAFEGUARD: Prevent division-by-zero by enforcing a tiny noise floor threshold
        psd_interp = np.maximum(psd_interp, 1e-30)

        whitened_fft = strain_fft / np.sqrt(psd_interp)
        strain_whitened = scipy.fft.irfft(whitened_fft, n=N)

        logging.info("Applying zero-phase Butterworth bandpass filter using Second-Order Sections (SOS)...")
        sos = scipy.signal.butter(8, [20, 300], btype='bandpass', fs=fs, output='sos')
        strain_filtered = scipy.signal.sosfiltfilt(sos, strain_whitened)

        # Crop strictly to 1.2s inspiral and 0.1s ringdown (1.3s total)
        logging.info("Cropping strictly to 1.3s window (1.2s inspiral + 0.1s ringdown)...")
        center_idx = N // 2
        start_idx = center_idx - int(1.2 * fs)
        end_idx = center_idx + int(0.1 * fs)
        chirp_isolated = strain_filtered[start_idx:end_idx]
        logging.info(f"Chirp isolated | Array shape: {chirp_isolated.shape}")

        # =====================================================================
        # PHASE 2: AUDIO DIGITAL SIGNAL PROCESSING (DSP)
        # =====================================================================
        logging.info("--- PHASE 2: AUDIO DSP EXPORT ---")

        # We need to stretch 1.3 seconds to exactly 15.0 seconds.
        stretch_rate = 1.3 / 15.0
        logging.info(f"Applying Phase Vocoder | Time-stretching 1.3s -> 15.0s (rate={stretch_rate:.4f})")
        audio_stretched = librosa.effects.time_stretch(
            y=chirp_isolated,
            rate=stretch_rate
        )

        n_steps_semitones = math.log2(10) * 12
        logging.info(f"Applying Phase Vocoder | Pitch shifting by +{n_steps_semitones:.1f} semitones (10x freq)")
        audio_shifted = librosa.effects.pitch_shift(
            y=audio_stretched,
            sr=fs,
            n_steps=n_steps_semitones,
            res_type='kaiser_best'
        )

        target_fs = 44100
        logging.info(f"Resampling to standard audio rate | {fs} Hz -> {target_fs} Hz")
        num_target_samples = int(len(audio_shifted) * target_fs / fs)
        audio_44100 = scipy.signal.resample(audio_shifted, num_target_samples)

        logging.info("Peak-normalizing and quantizing to 16-bit PCM integer space...")
        audio_normalized = audio_44100 / np.max(np.abs(audio_44100))
        audio_pcm16 = np.int16(audio_normalized * 32767)

        audio_filename = 'gw150914_shifted_chirp.wav'
        scipy.io.wavfile.write(audio_filename, target_fs, audio_pcm16)
        logging.info(f"SUCCESS: Audio exported to {audio_filename}")

        # =====================================================================
        # PHASE 3: HAPTIC EXPORT PIPELINE (UPDATED FOR DYNAMIC PWM)
        # =====================================================================
        logging.info("--- PHASE 3: HAPTIC DSP EXPORT ---")

        logging.info("Extracting analytic amplitude envelope via Hilbert transform...")
        analytic_signal = scipy.signal.hilbert(chirp_isolated)
        amplitude_envelope = np.abs(analytic_signal)

        # Apply a Savitzky-Golay filter to smooth out static spikes and track the true crescendo
        logging.info("Applying Savitzky-Golay filter to isolate the physical crescendo...")
        window_length = int(0.05 * fs)
        if window_length % 2 == 0:
            window_length += 1
        smoothed_envelope = scipy.signal.savgol_filter(amplitude_envelope, window_length, 3)
        smoothed_envelope = np.maximum(smoothed_envelope, 0) # Prevent negative dips

        # Downsample strictly to 20Hz (300 data points for 15 seconds)
        target_duration = 15.0
        target_hz = 20
        num_samples = int(target_duration * target_hz)
        logging.info(f"Time-dilating and downsampling envelope to exactly {target_hz}Hz ({num_samples} samples)...")
        haptic_envelope = scipy.signal.resample(smoothed_envelope, num_samples)

        logging.info("Normalizing to 1.0 max and applying Stevens' Power Law (gamma=0.5)...")
        haptic_envelope = haptic_envelope - np.min(haptic_envelope)
        haptic_envelope = haptic_envelope / np.max(haptic_envelope)
        perceptual_envelope = np.power(haptic_envelope, 0.5)

        # Clamp between 0.0 and 1.0 and round for clean JSON export
        perceptual_envelope = np.clip(perceptual_envelope, 0.0, 1.0)
        haptic_array = np.round(perceptual_envelope, 4).tolist()

        json_filename = 'gw150914_haptic_envelope.json'
        with open(json_filename, 'w') as f:
            json.dump(haptic_array, f)
        logging.info(f"SUCCESS: Haptic floating-point array exported to {json_filename} | Array Length: {len(haptic_array)}")

        # =====================================================================
        # PHASE 4: FRONTEND SYNCHRONIZATION METADATA
        # =====================================================================
        logging.info("--- PHASE 4: SYNCHRONIZATION METADATA ---")

        # Calculate exactly when the merger peak occurs in the 15-second stretched audio
        dilated_peak_time = 1.2 * (15.0 / 1.3) # ~13.846 seconds
        dilated_peak_time = round(dilated_peak_time, 3)

        metadata = {
            "duration": target_duration,
            "phases": {
                "inspiral": [0.0, dilated_peak_time],
                "merger": [dilated_peak_time, 14.200], # Roughly 350ms of peak collision chaos
                "ringdown": [14.200, target_duration]
            }
        }

        metadata_filename = 'metadata.json'
        with open(metadata_filename, 'w') as f:
            json.dump(metadata, f, indent=4)
        logging.info(f"SUCCESS: Metadata exported to {metadata_filename}")

        logging.info("--- PIPELINE COMPLETE ---")

    except Exception as e:
        logging.error(f"Pipeline Execution Failed: {str(e)}", exc_info=True)
        sys.exit(1)

if __name__ == "__main__":
    process_gw_data()

/usr/local/lib/python3.12/dist-packages/gwpy/time/_ligotimegps.py:42: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  from lal import LIGOTimeGPS
